# DiffLense on Colab: progressive graph super-resolution for CMS jets and CaloChallenge showers

This notebook runs the full [`superresolution-difflense`](https://github.com/rajveer43/superresolution-difflense) pipeline and saves every result to Google Drive:

`download → prepare → train_sr → generate → eval_sr → extrapolate → train_tagger → report`

| Stage | What it does |
|---|---|
| `download` | Fetches the raw data once to Drive: CERNBox for CMS jets, Zenodo for CaloChallenge. Complete files are skipped. |
| `prepare` | Builds HR truth, the noisy measured LR (sum-pool, noise, topo-clustering) and the train/val/test split, on local disk. |
| `train_sr` | Trains the graph flow-matching SR model on every scale transition that has truth. |
| `generate` | Runs SR on whole splits up to the native grid (128×128 jets, 16×9 showers). |
| `eval_sr` | Compares SR with HR truth using physics metrics (no PSNR/SSIM), and reproduces the paper's figures 6 and 7. |
| `extrapolate` | Continues from the native grid to 2048×2048. There is no truth there, so it checks energy closure, observable drift and spectra instead. |
| `train_tagger` | Trains a ResNet-18 on HR, LR, upsampled LR and SR inputs over several seeds: quark/gluon AUC for jets, energy resolution for showers. |
| `report` | Writes one markdown page with every table. |

**Every stage resumes.** After a disconnect, run the notebook from the top. Training continues from `last.pt` on Drive, generation continues from its last batch, and finished taggers are skipped. The local cache is rebuilt the same way each time, so the train/val/test split never changes.

**Before you start:** pick *Runtime → Change runtime type → GPU* (an A100 or L4 if you can; a T4 works with a smaller batch). The repo is private, so add a GitHub token as a Colab secret named `GITHUB_TOKEN` (key icon in the left bar; a fine-grained token with *Contents: read* on this repo is enough).

## 1. Settings
Every option lives in this one cell. Keep `SMOKE = True` for a first pass: a few minutes on a small subset, written to its own `smoke` folder, to check that everything runs. Then set `SMOKE = False` for the real run.

In [ ]:
DATASET   = "qg"               # "qg" = CMS quark/gluon jet images | "calo" = CaloChallenge DS2 electron showers
RUN_TAG   = "paper-baseline"   # short description; becomes part of the experiment folder name
SMOKE     = True               # True: quick end-to-end check on a small subset (separate folder)
RUN_DATE  = None               # None = today (YYYY-MM-DD). Set it to resume an experiment started on another day.

# --- real-run settings (ignored when SMOKE = True) ---
MAX_EVENTS        = 60000      # qg: jets used (139k available). calo: showers from dataset_2_1 (None = all 100k)
SR_EPOCHS         = 40         # early stopping usually ends sooner (patience 8); the paper used 100
SR_BATCH          = 25         # events per batch (paper). Use 12 on a T4 (16 GB) if you run out of memory
ENSEMBLE          = 10         # SR samples averaged per event (paper section 3.3)
STEPS             = 25         # ODE steps per sample (paper table 2)
GEN_TRAIN_MAX     = 20000      # train events super-resolved for the SR tagger (val and test are done in full)
TAGGER_SEEDS      = [42, 43, 44]
TAGGER_EPOCHS     = 30
EXTRAPOLATE_EVENTS = 32        # test events pushed all the way to 2048 x 2048

DRIVE_ROOT = "/content/drive/MyDrive/GSoC_DiffLense"
REPO_URL   = "github.com/rajveer43/superresolution-difflense.git"
BRANCH     = "main"

## 2. GPU check

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "no NVIDIA GPU visible")
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Switch the runtime to GPU: Runtime -> Change runtime type"

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 4. Get the code
Clones the repo on the first run and pulls the latest commit after that. The token is read from Colab Secrets and is never printed.

In [ ]:
import os, subprocess
REPO = "/content/superresolution-difflense"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO_URL}" if token else f"https://{REPO_URL}"

def git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token or "\0", "***")
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed:\n{out}\n(private repo: is the GITHUB_TOKEN secret set and "
                           "notebook access enabled?)")
    return out

if not os.path.exists(REPO):
    git("clone", "--branch", BRANCH, url, REPO)
else:
    git("pull", url, BRANCH, cwd=REPO)
COMMIT = git("rev-parse", "--short", "HEAD", cwd=REPO).strip()
print("code at commit", COMMIT)
print(git("log", "-1", "--format=%s (%cd)", cwd=REPO))

## 5. Install dependencies
Colab already has PyTorch, so only the remaining packages are installed (about 1 minute).

In [ ]:
!grep -v -E "^(torch|torchvision)" {REPO}/requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt

## 6. Experiment folders and the run helper
```
MyDrive/GSoC_DiffLense/
  datasets/<dataset>/raw/                    raw files, downloaded once and reused
  experiments/<dataset>/<date>_<run-tag>/    one experiment
      config.json  run_info.json  report.md
      logs/<stage>.log                       full console output of each stage
      sr_model/                              best.pt, last.pt, history.csv
      generated/                             SR outputs (resumable)
      evaluation/sr/  evaluation/extrapolation/
      taggers/
  exports/<dataset>_<date>_<run-tag>_results.zip
```
The preprocessed cache is too large for Drive, so it stays on local disk (`/content/data/cache`) and is rebuilt identically after a disconnect.

In [ ]:
import datetime, json, os, shutil, subprocess, sys
from IPython.display import Markdown, Image, display

date = RUN_DATE or datetime.date.today().isoformat()
tag = "smoke" if SMOKE else RUN_TAG
RUN_NAME = f"{date}_{tag}"
CONFIG = f"configs/{DATASET}.yaml"
EXP = f"{DRIVE_ROOT}/experiments/{DATASET}/{RUN_NAME}"
RAW_DRIVE = f"{DRIVE_ROOT}/datasets/{DATASET}/raw"
RAW_LOCAL = f"/content/data/raw/{DATASET}"
CACHE = f"/content/data/cache/{DATASET}" + ("_smoke" if SMOKE else "")
for d in (EXP, f"{EXP}/logs", RAW_DRIVE, RAW_LOCAL, f"{DRIVE_ROOT}/exports"):
    os.makedirs(d, exist_ok=True)

SETS = [f"run_name={RUN_NAME}", f"paths.out={EXP}", f"paths.cache={CACHE}", f"paths.raw={RAW_LOCAL}",
        f"paths.generated={EXP}/generated", "num_workers=2",
        f"sampling.dtype={'float16' if DATASET == 'qg' else 'float32'}"]  # shower cells can exceed fp16 range
if SMOKE:
    SETS += ["data.max_events=2000", "sr_train.epochs=2", "sr_train.physics_val_every=1",
             "sr_train.physics_val_events=32", "sr_train.max_val=200",
             "sampling.ensemble=2", "sampling.steps=5", "sampling.max_per_split={train: 600, val: 200, test: 300}",
             "refine.events=3", "refine.ensemble=2", "refine.steps=3", "refine.save_events=2",
             "tagger.seeds=[42]", "tagger.epochs=2", "tagger.n_boot=20",
             "eval.max_events=300", "eval.ensemble_study=[1,2,4]", "eval.ensemble_study_events=32"]
    if DATASET == "calo":
        SETS += ["data.test_max=500"]
else:
    SETS += [f"data.max_events={MAX_EVENTS}", f"sr_train.epochs={SR_EPOCHS}", f"sr_train.batch_size={SR_BATCH}",
             f"sampling.ensemble={ENSEMBLE}", f"sampling.steps={STEPS}",
             f"sampling.max_per_split={{train: {GEN_TRAIN_MAX}, val: null, test: null}}",
             f"tagger.seeds={TAGGER_SEEDS}", f"tagger.epochs={TAGGER_EPOCHS}", f"refine.events={EXTRAPOLATE_EVENTS}"]

def run(*stages, config=CONFIG, extra=(), log=None):
    """Run pipeline stages; stream the output here and append it to logs/<stage>.log on Drive."""
    cmd = [sys.executable, "-u", "-m", "difflense", *stages, "-c", config, "--set", *SETS, *extra]
    path = f"{EXP}/logs/{log or '_'.join(stages)}.log"
    with open(path, "a") as f:
        f.write(f"\n===== {datetime.datetime.now():%Y-%m-%d %H:%M:%S} | commit {COMMIT} | {' '.join(stages)}\n")
        p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end="")
            f.write(line)
        p.wait()
    if p.returncode:
        raise RuntimeError(f"stage(s) {stages} failed (exit {p.returncode}); full log: {path}")

# run_info.json: one entry per session (when, which code, which GPU, which settings)
info_path = f"{EXP}/run_info.json"
info = json.load(open(info_path)) if os.path.exists(info_path) else {
    "experiment": RUN_NAME, "dataset": DATASET, "started": datetime.datetime.now().isoformat(timespec="seconds"),
    "sessions": []}
info["sessions"].append({"time": datetime.datetime.now().isoformat(timespec="seconds"), "commit": COMMIT,
                         "gpu": torch.cuda.get_device_name(0), "smoke": SMOKE, "overrides": SETS})
json.dump(info, open(info_path, "w"), indent=2)
print("experiment folder:", EXP)

## 7. Download the dataset (once, to Drive)
- **qg:** the public CERNBox share `EYgmOkI9BjwxNqy`, which holds 3 parquet files of ML4Sci CMS jet images (139k jets, 3.7 GB).
- **calo:** Zenodo record 6366271, `dataset_2_1.hdf5` and `dataset_2_2.hdf5`, md5-checked.

This takes 5–15 minutes the first time. Later sessions skip files that are already complete. The files are then copied to local disk, which reads much faster than Drive.

In [ ]:
run("download", extra=[f"paths.raw={RAW_DRIVE}"])
!rsync -a --info=progress2 "{RAW_DRIVE}/" "{RAW_LOCAL}/"
!ls -lh "{RAW_LOCAL}"

## 8. Prepare the cache
This step builds the HR truth on the native grid and the noisy measured LR. The LR is sum-pooled 4×4 for jets or 4×3 for showers, then noise and 4-2-0 topo-clustering are applied. Splits are fixed by the seed, and scales come from the train split only. Expect about 5–10 minutes for 60k jets. If the cache already exists, this step is skipped.

In [ ]:
run("prepare")

## 9. Train the super-resolution model
This trains the graph transformer with conditional flow matching. It uses AdamW with warmup and cosine schedule, EMA weights, bf16 and gradient clipping, and stops early on the EMA validation loss. Every 5 epochs it also logs the event-energy resolution after the full ladder (`phys_sr_res` against `phys_lr_res`).

Checkpoints go to `sr_model/` on Drive after every epoch, so rerunning this cell resumes. Rough timing for 60k jets: about 3–6 minutes per epoch on an A100 and 3–4× that on a T4.

In [ ]:
run("train_sr")

In [ ]:
import pandas as pd
h = pd.read_csv(f"{EXP}/sr_model/history.csv")
display(h.tail(10))

## 10. Generate SR outputs (resumable)
This runs the ladder from the measured LR up to the native grid for every split, using the ensemble mean. The outputs are the inputs for the evaluation and the SR tagger. It is the slowest stage, since every event costs ensemble × steps × 2 forward passes per stage, so lower `GEN_TRAIN_MAX` if time is short. Progress is saved after every batch.

In [ ]:
run("generate")

## 11. Evaluate SR against the HR truth
Physics metrics on every grid that has truth, compared against the measured LR and the "spread uniformly" baseline:
- event energy residual and its resolution (paper fig. 6a), and resolution against ensemble size (fig. 6b);
- observable distributions and per-event residuals, including C2/C3/D2 (fig. 7);
- hit F1, cell-energy spectrum, per-event EMD, and ensemble calibration.

In [ ]:
run("eval_sr")

## 12. Progressive reconstruction to 2048 × 2048
The grids beyond the native one have **no ground truth**, so they are checked instead:
- **energy closure:** each grid must sum to the stage-1 energy, to round-off;
- **observable drift:** observables must stay stable as cells get finer;
- **power spectra** at each grid;
- **zoom panels:** the same physical window shown at every scale.

Refinement is adaptive and capped at `refine.max_nodes_per_event`. Section 16 measures how well the model handles a scale it never saw in training.

In [ ]:
run("extrapolate")

## 13. Downstream ResNet-18 taggers
One ResNet-18 is trained per input (HR, measured LR, LR uniformly upsampled, SR) and per seed, all on the same events.
- **qg:** quark/gluon ROC AUC and background rejection 1/ε_B at ε_S = 30/50/70 %.
- **calo:** response bias and resolution of the incident-energy regression.

The HR-trained network is also scored directly on SR to expose any domain shift. Finished runs are skipped when you rerun.

In [ ]:
run("train_tagger")

## 14. Report

In [ ]:
run("report")
display(Markdown(open(f"{EXP}/report.md").read()))

## 15. Key figures

In [ ]:
import glob
def show(pattern, width=900):
    for f in sorted(glob.glob(f"{EXP}/{pattern}")):
        print(os.path.relpath(f, EXP))
        display(Image(f, width=width))

show("evaluation/sr/figures/energy_residual_total.png", 500)
show("evaluation/sr/figures/ensemble_size.png", 450)
show("evaluation/sr/figures/event_0.png")
show("evaluation/sr/figures/observables_*.png")
show("evaluation/sr/figures/energy_correlators_*.png")
show("evaluation/extrapolation/figures/zoom_event_0_*.png")
show("evaluation/extrapolation/figures/power_spectra.png", 500)
show("evaluation/extrapolation/figures/drift_*.png", 450)
show("taggers/figures/roc.png", 500)

## 16. (Optional) Held-out-scale check: how good is the extrapolation?
This trains a second model **without** the last transition that has truth (64→128 for jets, 8×9→16×9 for showers), then scores that stage against truth. The gap between this experiment's metrics on that grid and the main run's estimates the error of the stages beyond native, which can't be measured directly. It is saved as its own experiment, `<date>_<run-tag>-extrapolation-check`, and reuses the same cache.

In [ ]:
RUN_CHECK = False
if RUN_CHECK:
    EXP_MAIN = EXP
    check_name = f"{RUN_NAME}-extrapolation-check"
    EXP = f"{DRIVE_ROOT}/experiments/{DATASET}/{check_name}"
    os.makedirs(f"{EXP}/logs", exist_ok=True)
    SETS_MAIN = list(SETS)
    SETS = [s for s in SETS if not s.startswith(("run_name=", "paths.out=", "paths.generated="))] + [
        f"run_name={check_name}", f"paths.out={EXP}", f"paths.generated={EXP}/generated"]
    try:
        run("train_sr", "generate", "eval_sr", "report", config=f"configs/{DATASET}_extrapolation_check.yaml",
            extra=["eval.ensemble_study=[]"])
        display(Markdown(open(f"{EXP}/report.md").read()))
    finally:
        EXP, SETS = EXP_MAIN, SETS_MAIN

## 17. Export a results bundle
This zips the experiment into `exports/` on Drive: configs, logs, metrics, CSVs, figures, the report, the SR model's `best.pt` and the sparse 2048×2048 events. The large generated arrays and the per-epoch `last.pt` checkpoints are left out.

In [ ]:
import zipfile
zip_path = f"{DRIVE_ROOT}/exports/{DATASET}_{RUN_NAME}_results.zip"
skip_dirs = {"generated"}
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(EXP):
        dirs[:] = [d for d in dirs if d not in skip_dirs]
        for f in files:
            p = os.path.join(root, f)
            rel = os.path.relpath(p, EXP)
            if f == "last.pt" or (f == "best.pt" and rel.startswith("taggers")):
                continue
            z.write(p, os.path.join(f"{DATASET}_{RUN_NAME}", rel))
print(f"{zip_path}  ({os.path.getsize(zip_path) / 1e6:.1f} MB)")